**Importing The Dependencies**

In [98]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRFClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

## **1. Data Collection & Loading**

In [99]:
# load the clean dataset into a pandas DataFrame
df = pd.read_csv("../dataset/clean dataset/final_telco_customer_churn_cleaned_dataset.csv")

In [100]:
df.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,...,contract,paper_less_billing,payment_method,monthly_charges,total_charges,churn,customer_profile,churn_numeric,tenure_groups,total_services
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,Partner Only,0,0-1 Year,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,Single / No Family,0,2-4 Years,4
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Single / No Family,1,0-1 Year,4
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,Single / No Family,0,2-4 Years,4
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Single / No Family,1,0-1 Year,2


In [101]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   gender              7043 non-null   object 
 1   senior_citizen      7043 non-null   int64  
 2   partner             7043 non-null   object 
 3   dependents          7043 non-null   object 
 4   tenure              7043 non-null   int64  
 5   phone_service       7043 non-null   object 
 6   multiple_lines      7043 non-null   object 
 7   internet_service    7043 non-null   object 
 8   online_security     7043 non-null   object 
 9   online_backup       7043 non-null   object 
 10  device_protection   7043 non-null   object 
 11  tech_support        7043 non-null   object 
 12  streaming_tv        7043 non-null   object 
 13  streaming_movies    7043 non-null   object 
 14  contract            7043 non-null   object 
 15  paper_less_billing  7043 non-null   object 
 16  paymen

In [102]:
# drop these columns since they may cause data leakage
df = df.drop(columns = ['churn_numeric', 'tenure_groups','customer_profile'])

In [103]:
df.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,...,tech_support,streaming_tv,streaming_movies,contract,paper_less_billing,payment_method,monthly_charges,total_charges,churn,total_services
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,2
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,4
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,4
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,4
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2


## **3. Data Splitting**

**Splitting Data into Features and Target**

In [104]:
# splitting the features and target
X = df.drop(columns = ["churn"])
Y = df["churn"]

**Splitting Data Into Training And Testing Data**

In [105]:
# split training and testing data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state = 2, stratify=Y)

In [106]:
print(X.shape, X_train, X_test)

(7043, 20)       gender  senior_citizen partner dependents  tenure phone_service  \
5406    Male               0     Yes        Yes      71           Yes   
504     Male               0      No        Yes      69           Yes   
874     Male               0      No        Yes      45           Yes   
5411  Female               1     Yes         No       1            No   
3520  Female               0     Yes         No      16           Yes   
...      ...             ...     ...        ...     ...           ...   
6846  Female               0      No         No       1           Yes   
5102  Female               0     Yes         No      72           Yes   
2926  Female               0     Yes         No      69           Yes   
2063    Male               0     Yes        Yes      51           Yes   
4309  Female               0      No         No      22            No   

        multiple_lines internet_service      online_security  \
5406               Yes      Fiber optic         

In [107]:
print(Y_train.shape)
print(Y_train.value_counts())

(5634,)
churn
No     4139
Yes    1495
Name: count, dtype: int64


## **4. Data Preprocessing**

**Label Encoding Of Target Column**

In [108]:
# load the Label Encoder
label_encoder = LabelEncoder()

# encode target(y) using test using LabelEncoder
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_test_encoded = label_encoder.transform(Y_test)

print(Y_train_encoded)

[0 0 0 ... 0 0 0]


**One Hot Encoding Of Feature Column**

In [109]:
# Define categorical feature groups
cats_col = ['gender', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'streaming_tv', 'streaming_movies', 'contract', 'paper_less_billing', 'payment_method']

ohe_encoder = OneHotEncoder(drop='first', sparse_output=False)

X_train_encoded = ohe_encoder.fit_transform(X_train[cats_col])
X_test_encoded = ohe_encoder.transform(X_test[cats_col])

encoded_cols = ohe_encoder.get_feature_names_out(cats_col)

X_train_encoder_df = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train.index)
X_test_encoder_df = pd.DataFrame(X_test_encoded, columns=encoded_cols, index=X_test.index)

**Standard Scaler**

In [110]:
num_cols = ['tenure', 'senior_citizen', 'monthly_charges', 'total_charges']

scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train[num_cols])
X_test_scale = scaler.transform(X_test[num_cols])

X_train_scale_df = pd.DataFrame(X_train_scale, columns= num_cols, index=X_train.index)
X_test_scale_df = pd.DataFrame(X_test_scale, columns= num_cols, index=X_test.index)

X_train_final = pd.concat([X_train_scale_df, X_train_encoder_df], axis=1)
X_test_final = pd.concat([X_test_scale_df, X_test_encoder_df], axis=1)

print("Data is now encoded and scaled. Preview of first 5 rows")
X_train_final.head()

Data is now encoded and scaled. Preview of first 5 rows


,tenure,senior_citizen,monthly_charges,total_charges,gender_Male,partner_Yes,dependents_Yes,phone_service_Yes,multiple_lines_No phone service,multiple_lines_Yes,...,streaming_tv_No internet service,streaming_tv_Yes,streaming_movies_No internet service,streaming_movies_Yes,contract_One year,contract_Two year,paper_less_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check
5406,1.571838,-0.441199,1.090841,2.055052,1.0,1.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
504,1.490442,-0.441199,0.498992,1.431865,1.0,0.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
874,0.513698,-0.441199,1.174200,1.047528,1.0,0.0,1.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
5411,-1.277000,2.266551,-1.323236,-0.993040,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3520,-0.666535,-0.441199,-1.496623,-0.864817,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


**Synthetic Minority Oversampling Technique(SMOTE)**

In [111]:
smote = SMOTE(random_state=2)

## **5. Model Selection & Development**

In [112]:
# list of models 

models = [LogisticRegression(random_state=2), RandomForestClassifier(random_state = 2), DecisionTreeClassifier(random_state = 2), XGBRFClassifier(random_state = 2)]

def compare_models_cross_validation():
    # Cross validating with default hyperparameters
    for model in models:

        imb_pipeline = make_pipeline(smote, model)

        cv_score = cross_val_score(imb_pipeline, X_train_final, Y_train_encoded, cv=5)
        f1_cv_score = cross_val_score(imb_pipeline, X_train_final, Y_train_encoded, cv=5, scoring='f1')
        mean_accuracy = round(sum(cv_score)/len(cv_score),2)
        f1_mean_accuracy = round(sum(f1_cv_score)/len(f1_cv_score),2)
        

        # CLean up the model name string for printing
        model_name = model.__class__.__name__

        print(f"Cross-Validation Accuracies for the {model_name} : {cv_score}")
        print(f"Accuracy Score of {mean_accuracy}%")
        print(f"Mean F1 Score of {f1_mean_accuracy}%")
        print("*" * 150)

compare_models_cross_validation()

Cross-Validation Accuracies for the LogisticRegression : [0.74179237 0.75865129 0.75421473 0.73824312 0.75044405]
Accuracy Score of 0.75%
Mean F1 Score of 0.63%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the RandomForestClassifier : [0.76308784 0.79148181 0.76929902 0.77639752 0.79129663]
Accuracy Score of 0.78%
Mean F1 Score of 0.58%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the DecisionTreeClassifier : [0.69831411 0.73469388 0.7133984  0.72937001 0.73712256]
Accuracy Score of 0.72%
Mean F1 Score of 0.51%
******************************************************************************************************************************************************
Cross-Validation Accuracies for the XGBRFClassifier : [0

**For this dataset, XGBRF Classifier has the highest accuracy and F-1 score wtih default hyperparamters compared to other models.**

In [113]:
xgb_model = XGBRFClassifier(random_state=2, eval_metric = 'logloss')

In [114]:
tuning_pipeline = make_pipeline(smote, xgb_model)

**Applying GridSearchCV**

In [115]:
hyperparameter_grid = {
        'xgbrfclassifier__n_estimators' : [10, 50, 100],
        'xgbrfclassifier__max_depth' : [3, 5, 7],
        'xgbrfclassifier__learning_rate' : [0.05, 0.1, 0.2]
    }

grid_search = GridSearchCV(estimator = tuning_pipeline, param_grid = hyperparameter_grid, cv=5, scoring = 'f1')

print("Optimizing XGBoost Random Forest parameters...")
grid_search.fit(X_train_final, Y_train_encoded)

print('=' * 120)
print(f"Best Hyperparamters: {grid_search.best_params_}")
print(f"Optimized Cross-Validation F-1 Score: {grid_search.best_score_}")
print("=" * 120)

Optimizing XGBoost Random Forest parameters...
Best Hyperparamters: {'xgbrfclassifier__learning_rate': 0.05, 'xgbrfclassifier__max_depth': 7, 'xgbrfclassifier__n_estimators': 50}
Optimized Cross-Validation F-1 Score: 0.6264577854283282


## **6. Model Training & Evaluation**

**Model Training**

In [116]:
final_model = XGBRFClassifier(random_state = 2, learning_rate = 0.05, max_depth = 7, n_estimators = 100)

In [117]:
# Connecting SMOTE abd XGBRFCLassifier model with best parameters together in final training pipeline
pipeline = make_pipeline(smote, final_model)

# train the pipeline on the training data
pipeline.fit(X_train_final, Y_train_encoded)

,steps,"[('smote', ...), ('xgbrfclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,'auto'
,random_state,2
,k_neighbors,5
,learning_rate,0.05
,subsample,0.8
,colsample_bynode,0.8
,reg_lambda,1e-05


**Model Evaluation**

In [118]:
# evaluate on test data
Y_test_pred = pipeline.predict(X_test_final)

print("Accuracy Score:\n", accuracy_score(Y_test_encoded, Y_test_pred))
print('=' * 60)
print("Confusion Matrix:\n", confusion_matrix(Y_test_encoded, Y_test_pred))
print('=' * 60)
print(" Classification Report:\n", classification_report(Y_test_encoded, Y_test_pred))
print('=' * 60)

Accuracy Score:
 0.7778566359119943
Confusion Matrix:
 [[806 229]
 [ 84 290]]
 Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.78      0.84      1035
           1       0.56      0.78      0.65       374

    accuracy                           0.78      1409
   macro avg       0.73      0.78      0.74      1409
weighted avg       0.81      0.78      0.79      1409



**Feature Importance**

## **7. Save The Model**

In [119]:
# save the XGBRGClassifier model
trained_model_filename = "../models/trained_model.sav"

with open(trained_model_filename, 'wb') as f:
    pickle.dump(final_model, f)

In [120]:
# save the LabelEncoder model
label_encoder_filename = "../models/label_encoder.sav"

with open(label_encoder_filename, 'wb') as f:
    pickle.dump(label_encoder, f)

In [121]:
# save the StandardScaler model
scaler_filename = "../models/standardscaler.sav"

with open(scaler_filename, 'wb') as f:
    pickle.dump(scaler, f)

In [122]:
# save the OneHotEcoder model
one_hot_encoder_filename = "../models/one_hot_encoder.sav"

with open(one_hot_encoder_filename, 'wb') as f:
    pickle.dump(ohe_encoder, f)